In [ ]:
# Normalizing

## Why Normalize?

Data normalization solves a fundamental problem: features measured on different scales create **implicit biases** in any algorithm that computes distances, dot products, or gradients. A feature ranging over [0, 1,000,000] will dominate one ranging over [0, 1] even if they are equally informative.

---

## The Core Methods

### Min-Max Scaling (Feature Scaling)

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Maps all values to $[0, 1]$. The transformation is an **affine map** — it preserves the shape and relative distances of the distribution but does not change its form (a skewed distribution stays skewed).

**Geometric view:** You are stretching/compressing the axis so the range has unit length. Every pairwise ratio of distances is preserved.

**When to use:** When you know the true bounds and outliers are not a concern. Pixel intensities (bounded [0,255]) are a classic example.

**Sensitivity to outliers:** High. One extreme value compresses everything else into a narrow band.

---

### Z-score Standardization (Standard Scaling)

$$x' = \frac{x - \mu}{\sigma}$$

Produces a distribution with **zero mean and unit variance**. This is the most theoretically grounded normalization for most ML contexts.

**Why it works:** The standardized variable $x'$ is dimensionless. You have removed the first two moments of the distribution, so algorithms operate on the *structure* of the data, not its scale or location.

**Connection to the Normal distribution:** If $x \sim \mathcal{N}(\mu, \sigma^2)$, then $x' \sim \mathcal{N}(0, 1)$. For non-normal data the mean and variance are still removed — the distribution shape is preserved but not Gaussianized.

**Effect on gradient descent:** The loss landscape becomes more spherically symmetric. Without standardization, the condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ of the Hessian can be enormous, forcing tiny learning rates and slow convergence. Standardization roughly equalizes the curvature across dimensions.

$$\kappa_{\text{scaled}} \approx \frac{\max \sigma_i^2}{\min \sigma_i^2} \ll \kappa_{\text{raw}}$$

---

### L2 (Unit Norm) Normalization — Per Sample

$$x' = \frac{x}{\|x\|_2}$$

Projects each sample onto the **unit hypersphere** $S^{d-1}$. This discards magnitude entirely and encodes only direction.

**Geometric view:** All samples lie on the same sphere. Euclidean distance between two unit vectors equals $\sqrt{2(1 - \cos\theta)}$, so it is a monotone function of the angle between them. L2 normalization makes Euclidean distance equivalent to cosine similarity — useful when relative patterns matter more than absolute magnitudes (e.g., TF-IDF vectors, embeddings).

**L1 normalization** ($x' = x / \|x\|_1$) maps samples to the unit simplex — the components sum to 1, making them interpretable as proportions.

---

### Robust Scaling

$$x' = \frac{x - \text{median}(x)}{\text{IQR}(x)}$$

Uses the median and interquartile range instead of mean and standard deviation. Because the median and IQR are **breakdown-point-50%** statistics, up to 50% of the data can be extreme values without affecting the result — far more robust than z-score scaling (breakdown point ≈ 0).

---

## Theoretical Comparisons

| Method | Preserves | Removes | Robust to outliers |
|---|---|---|---|
| Min-Max | Shape, distance ratios | Offset, scale | No |
| Z-score | Shape | Mean, variance | No |
| Unit norm (L2) | Direction | Magnitude | Partial |
| Robust scaling | Rank structure | Median, spread | Yes |

---

## Normalization vs. Standardization vs. Whitening

These terms are often conflated:

- **Normalization** — any transformation that maps data to a standard range or unit norm.
- **Standardization** — specifically removing mean and scaling to unit variance (z-score).
- **Whitening (sphering)** — goes further: removes *all* linear correlations between features so the covariance matrix becomes the identity $\Sigma = I$.

  $$X_{\text{white}} = X W^{-1/2}$$

  where $W = \text{diag}(\lambda_1, \ldots, \lambda_d)$ from PCA. Whitening is optimal for algorithms that assume $\mathcal{N}(0, I)$ inputs (e.g., ICA), but can amplify noise in low-variance directions.

---

## Which Algorithms Care Most

**Distance-based** (KNN, K-means, SVM with RBF kernel): Extremely sensitive. Unscaled features make distance metrics meaningless.

**Gradient-based** (linear regression, logistic regression, neural networks): Standardization dramatically accelerates convergence by conditioning the optimization landscape.

**Tree-based** (decision trees, random forests, XGBoost): Invariant to monotone feature transformations — scaling has *no effect* on splits or predictions.

**PCA / SVD**: Must standardize first or the principal components will align with the highest-variance features purely due to scale, not information content.

---

## Distributional Transforms (Beyond Scale)

When a feature is heavily skewed, scaling addresses *range* but not *shape*. True Gaussianization requires nonlinear transforms:

- **Log transform:** $x' = \log(x+1)$ — compresses heavy right tails, appropriate for count data and multiplicative processes.
- **Box-Cox:** $x' = (x^\lambda - 1)/\lambda$ — parameterized power transform that finds the $\lambda$ maximizing normality (MLE). Requires $x > 0$.
- **Yeo-Johnson:** Extension of Box-Cox to $x \in \mathbb{R}$.
- **Quantile transform:** Maps to a target distribution (uniform or normal) by matching empirical CDF to the target CDF. Fully nonparametric but destroys distance relationships.

---

## Leakage Warning

**Always fit scaler parameters on the training set only.** Fitting on the full dataset leaks test-set statistics ($\mu$, $\sigma$, $x_{\min}$, $x_{\max}$) into the training process, causing optimistically biased evaluation. In a pipeline:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
pipe.fit(X_train, y_train)   # scaler sees only training data
pipe.score(X_test, y_test)   # test data transformed with train stats
```


# Encoding Categorical Variables

Most ML algorithms operate on real-valued tensors. Categorical variables — nominal (unordered) or ordinal (ordered) — must be mapped to numbers without introducing spurious structure. The choice of encoding is a modeling decision with real consequences.

---

## Label Encoding (Integer Encoding)

Map each category to an integer: $\{A, B, C\} \to \{0, 1, 2\}$.

**When it's valid:** Only for **ordinal** variables where the integer ordering is meaningful (e.g., `low=0, medium=1, high=2`).

**When it's wrong:** For nominal variables, integer encoding implies $A < B < C$ and $|A - C| = 2|A - B|$. A linear model or distance-based algorithm will treat this as genuine numeric structure, introducing bias that has nothing to do with the data.

---

## One-Hot Encoding (OHE)

For a variable with $K$ categories, create $K$ binary columns, exactly one of which is 1.

$$\text{Color} = \text{Red} \Rightarrow [1, 0, 0], \quad \text{Green} \Rightarrow [0, 1, 0], \quad \text{Blue} \Rightarrow [0, 0, 1]$$

**No ordinal assumption.** Each category gets its own axis in a $K$-dimensional space — all categories are equidistant from each other (the columns are orthogonal).

**The dummy variable trap:** The $K$ columns sum to 1, so they are perfectly multicollinear. Any linear model with an intercept is rank-deficient. Drop one column (**dummy coding**, $K-1$ dummies) to break the collinearity. The dropped category becomes the reference level — all other coefficients are interpreted relative to it.

Tree-based models are immune to multicollinearity and can use all $K$ columns.

**High cardinality:** A variable with 10,000 unique values produces a 10,000-wide sparse matrix. OHE becomes impractical — use target encoding, hashing, or embeddings instead.

---

## Dummy Coding vs. Effect Coding

Both use $K-1$ columns but differ in the reference category treatment:

| | Dummy coding | Effect coding (sum-to-zero) |
|---|---|---|
| Reference category | All zeros | All $-1$s |
| Intercept meaning | Mean of reference group | Grand mean |
| Coefficient meaning | Deviation from reference | Deviation from grand mean |

Effect coding is preferred in ANOVA-style analysis where deviations from the overall mean are more interpretable than deviations from an arbitrarily chosen reference.

---

## Ordinal Encoding

For ordinal variables, assign integers that respect the ordering. Unlike label encoding applied blindly, ordinal encoding is *intentional* — you are asserting that the gaps are meaningful or at least monotone.

If the gaps between levels are unequal (e.g., `cold=0, warm=5, hot=6`), custom mappings are more honest than uniform integer steps.

---

## Target Encoding (Mean Encoding)

Replace each category with the mean of the target variable $y$ within that category:

$$\hat{x}_k = \mathbb{E}[y \mid X = k] \approx \frac{\sum_{i: x_i = k} y_i}{n_k}$$

**Why it works:** Compresses cardinality to a single real number that is directly informative about the target. Handles high-cardinality variables elegantly.

**The overfitting problem:** For rare categories ($n_k$ small), the in-sample mean is noisy and will overfit. The model memorizes the training label for that rare category. Solutions:

- **Smoothing (shrinkage toward the prior):**
$$\hat{x}_k = \frac{n_k \cdot \bar{y}_k + \lambda \cdot \bar{y}_{\text{global}}}{n_k + \lambda}$$
  where $\lambda$ controls how much rare categories are pulled toward the global mean. As $n_k \to \infty$, the estimate converges to the within-category mean; as $n_k \to 0$, it converges to the global mean.

- **Cross-fold target encoding:** Compute category means on out-of-fold data (never on the same rows being encoded). This is the standard approach in production — it is mathematically equivalent to leave-one-out encoding and prevents leakage.

**Leakage:** Computing target encoding on the full training set before a cross-validation split leaks future information. Always encode within the CV loop.

---

## Binary Encoding

Encode the integer label of each category in binary: category $k$ → $\lceil \log_2 K \rceil$ bits.

Reduces dimensionality from $K$ (OHE) to $\log_2 K$ columns. Introduces weak ordinal structure (categories share bit patterns), which is acceptable for tree-based models but problematic for linear models. A middle ground between OHE and label encoding.

---

## Hashing (Feature Hashing / Hashing Trick)

Map categories to a fixed-width vector of size $m$ using a hash function $h$:

$$\phi(x)_j = \sum_{k : h(k) = j} x_k$$

Collisions (two categories mapping to the same bucket) introduce noise but the error is bounded and often acceptable. **Advantages:** Fixed memory regardless of cardinality, handles unseen categories at test time gracefully, no vocabulary needed. Used extensively in NLP (bag-of-words) and online learning.

---

## Embeddings (Learned Encodings)

Rather than fixing the encoding before training, learn a dense $d$-dimensional vector for each category jointly with the model (entity embeddings). Used in:

- **Neural networks:** An embedding layer maps category index → trainable $\mathbb{R}^d$ vector.
- **Word2Vec / fastText / GloVe:** Pre-trained embeddings for text tokens.
- **Tabular deep learning (TabNet, FT-Transformer):** Learned embeddings for categorical features in tabular data.

Embeddings capture *semantic structure* — categories that appear in similar contexts get similar vectors. The dimensionality $d$ is a hyperparameter; a common heuristic is $d \approx \min(50, \lceil K/2 \rceil)$.

---

## Handling Unseen Categories at Test Time

| Method | Unseen category behavior |
|---|---|
| OHE (sklearn) | Error or all-zeros (configurable) |
| Label encoding | Error unless `handle_unknown` set |
| Target encoding | Falls back to global mean |
| Hashing | Hash collision — graceful degradation |
| Embeddings | No embedding exists — needs fallback (UNK token, random init) |

Always account for distribution shift: a category present at test time but not at train time is a real-world occurrence, not an edge case.

---

## Frequency / Count Encoding

Replace each category with its frequency (or count) in the training set:
$$\hat{x}_k = P(X = k) = \frac{n_k}{n}$$

No leakage risk, no smoothing needed. Useful when rare categories genuinely deserve low weight. Loses the ability to distinguish two categories that happen to have the same frequency.

---

## Encoding Strategy by Algorithm

| Algorithm | Recommended encoding |
|---|---|
| Linear / logistic regression | OHE (drop one, watch for multicollinearity) |
| SVM | OHE |
| KNN | OHE (distances are meaningful) |
| Tree-based (RF, XGBoost, LightGBM) | Ordinal / label encoding, or native categorical support |
| Neural networks | Embeddings for high cardinality; OHE for low |
| High-cardinality nominal | Target encoding (with CV), hashing, or embeddings |
| Ordinal variables | Ordinal encoding with meaningful integer mapping |

**LightGBM and CatBoost** have native categorical support — they find optimal splits over categories directly without any pre-encoding, which is theoretically superior to OHE for tree models.


# Feature Engineering

Feature engineering is the process of using domain knowledge and mathematical insight to construct representations of raw data that make the underlying structure more accessible to a learning algorithm. The Vapnik-Chervonenkis perspective: a good feature representation reduces the effective VC dimension needed to solve the problem, allowing a simpler hypothesis class to achieve low bias.

---

## Why Features Matter More Than Models

The No Free Lunch theorem guarantees that no algorithm is universally superior. But the *representation* of the problem is not equally arbitrary — the right features can make a linear model outperform a deep neural net on a given task. Much of the historical progress in ML competitions was driven by feature engineering, not model selection.

The fundamental insight: models learn functions of inputs. If the true function $f^*(x)$ is a simple function of some transformation $\phi(x)$, then $f^* \circ \phi^{-1}$ is simple in the transformed space. The engineer's job is to find $\phi$.

---

## Mathematical Transformations

### Nonlinear Rescaling

When a feature has an exponential or multiplicative relationship with the target, a log transform linearizes it:
$$x' = \log(x + 1)$$
This makes the relationship accessible to linear models without adding interaction terms. Similarly:
- $\sqrt{x}$: stabilizes variance for count data (Poisson processes have $\text{Var}(X) = \mu$, so $\sqrt{X}$ has approximately constant variance)
- $x^2$, $x^3$: amplify large values, useful when high-end behavior dominates
- $1/x$: appropriate for rates (e.g., converting time-to-event into a rate)

### Polynomial Features

Explicitly generate interactions and higher-order terms:
$$[x_1, x_2] \to [1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

A degree-$d$ polynomial expansion of $p$ features produces $\binom{p+d}{d}$ features — grows combinatorially. Allows a linear model to fit any polynomial decision boundary. The kernel trick (polynomial kernel) computes this inner product implicitly in $O(p)$ rather than materializing the expanded space.

### Binning (Discretization)

Map a continuous variable to a discrete bin index. Useful when:
- The relationship with the target is non-monotone (piecewise constant is a better approximation)
- The variable has clear natural breakpoints (age groups, income brackets)
- You want to reduce sensitivity to measurement noise

**Equal-width bins:** Fixed interval size. Sensitive to outliers.
**Equal-frequency (quantile) bins:** Each bin contains the same number of samples. More robust, better-calibrated for skewed distributions.
**Decision tree binning:** Find optimal split points by fitting a shallow tree on the single variable — the split thresholds are the bin boundaries. This is essentially how gradient boosting learns.

---

## Interaction Features

An interaction between $x_i$ and $x_j$ captures effects that cannot be explained by either feature alone:
$$x_{ij} = x_i \cdot x_j$$

**Example:** The effect of temperature on ice cream sales depends on whether it's a weekend. Neither temperature nor weekend alone fully explains sales — their product does.

For linear models, interactions must be constructed explicitly. Tree-based models discover interactions automatically through multi-level splits, which is one reason they often outperform linear models on tabular data without manual feature engineering.

**Ratio features:** $x_i / x_j$ — normalizes one variable by another, capturing relative magnitude. Examples: price-to-earnings ratio, debt-to-income, clicks-per-impression. Ratios compress two correlated features into one that captures their relationship directly.

---

## Temporal Features

Raw timestamps carry no direct numeric meaning useful to most models. Decompose into:

- **Cyclical encoding:** Hours (0–23), months (1–12), and day-of-week are circular — hour 23 and hour 0 are adjacent. Encoding as integers imposes a false discontinuity. Use sine/cosine projections:
$$x_{\sin} = \sin\left(\frac{2\pi \cdot x}{T}\right), \quad x_{\cos} = \cos\left(\frac{2\pi \cdot x}{T}\right)$$
  where $T$ is the period. The two columns together encode position on a circle, preserving proximity across the boundary.

- **Calendar features:** Year, quarter, month, week-of-year, day-of-month, day-of-week, hour, is_weekend, is_holiday. Each captures a different periodicity.
- **Lag features:** $x_{t-1}, x_{t-2}, \ldots$ — the value of the series at previous time steps. The core representation for time-series models.
- **Rolling statistics:** Rolling mean, rolling std, rolling min/max over a window $w$. Smooth short-term noise while preserving trend.
- **Time since event:** Days since last purchase, days since last login. Captures recency effects directly.

---

## Spatial Features

- **Haversine distance** between coordinates (accounts for Earth's curvature)
- **Distance to nearest landmark** (store, hospital, transit stop)
- **Geohash / H3 grid encoding:** Map coordinates to a hierarchical grid cell — coarser cells generalize more, finer cells are more precise. Converts continuous coordinates into a categorical feature suitable for target encoding.
- **Cluster assignment:** Run k-means on spatial coordinates, use cluster label or distance to centroid as features.

---

## Text Features

- **Bag of Words (BoW):** Count or binary occurrence of each vocabulary token. Sparse, high-dimensional, ignores word order.
- **TF-IDF:** $\text{tf-idf}(t, d) = \text{tf}(t,d) \cdot \log\frac{N}{df(t)}$ — down-weights terms that appear in many documents (low discriminative power) and up-weights rare informative terms.
- **N-grams:** Sequences of $n$ consecutive tokens. Bigrams and trigrams capture local word order and phrases that BoW misses.
- **Pre-trained embeddings:** Map documents to dense vectors via mean-pooling word vectors (Word2Vec, GloVe) or transformer encoders (BERT, sentence-transformers). Captures semantic similarity.

---

## Aggregate and Group Features

For relational or grouped data, compute statistics within groups and join back to the row level:

- Mean/median/std of the target within a group (with leakage controls — same as target encoding)
- Count of events per entity within a time window
- Rank of the row within its group
- Deviation from group mean: $x_i - \bar{x}_{\text{group}}$

These are sometimes called **entity-level features** or **group statistics** and are extremely powerful for problems with repeated measurements per entity (customers, users, products).

---

## Feature Selection as a Form of Engineering

Not all engineered features will be useful. Adding irrelevant features:
- Increases dimensionality (curse of dimensionality for distance-based methods)
- Adds noise that hurts generalization
- Slows training

Formal selection methods:
- **Filter methods:** Score each feature independently (mutual information, $\chi^2$, ANOVA F-statistic, Pearson correlation). Fast, model-agnostic, ignores interactions.
- **Wrapper methods:** Evaluate feature subsets by model performance (recursive feature elimination, forward/backward selection). Captures interactions but expensive.
- **Embedded methods:** Regularization-induced sparsity (Lasso / $\ell_1$) or tree feature importances. Efficient and model-aware.
- **Permutation importance:** Shuffle one feature at a time and measure the drop in model performance. Model-agnostic, captures the feature's actual contribution including interactions.

---

## Leakage in Feature Engineering

Feature leakage is the most dangerous failure mode: a feature that encodes information not available at prediction time, making training performance meaningless.

**Common sources:**
- **Target leakage:** A feature that is causally downstream of the target (e.g., using a post-treatment measurement to predict treatment outcome)
- **Temporal leakage:** Using future data to construct features for a past prediction (e.g., computing a 7-day rolling mean without anchoring it at $t - 7$)
- **Group leakage:** Computing aggregate statistics on the full dataset before splitting (e.g., target encoding without cross-folding)
- **Preprocessing leakage:** Fitting a scaler, imputer, or encoder on the full dataset before the CV split

**Detection:** A suspiciously high training/validation performance gap, or a feature that becomes by far the most important — investigate its construction.

---

## Automated Feature Engineering

**Featuretools:** Applies a library of aggregation and transformation primitives across relational table joins — effectively automates the group aggregation patterns above.

**Deep feature synthesis (DFS):** Stacks primitives to generate features like "mean of sum of X grouped by Y joined to Z" — combinatorial but filtered by relevance.

These tools generate candidate features that are still subject to selection and leakage review. They accelerate exploration but do not replace domain knowledge.

---

## Summary: Feature Engineering Mindset

| Goal | Technique |
|---|---|
| Linearize a nonlinear relationship | Log, sqrt, power transforms |
| Capture multiplicative interactions | Product and ratio features |
| Make circular variables continuous | Sine/cosine encoding |
| Capture temporal patterns | Lags, rolling stats, calendar decomposition |
| Handle high-cardinality categoricals | Target encoding, embeddings, hashing |
| Reduce a skewed distribution | Log, Box-Cox, quantile transform |
| Encode proximity/clustering | Distance features, geohash, cluster assignment |
| Capture group-level behavior | Aggregate statistics (with leakage controls) |
| Reduce noise from irrelevant features | Feature selection (filter, wrapper, embedded) |


# Cross-Validation & Model Evaluation

Model evaluation answers two distinct questions: *how well does this model generalize to unseen data*, and *which of several candidate models generalizes best*. Both require an honest estimate of out-of-sample performance — one that has not been inflated by the choices made during training.

---

## The Bias-Variance Decomposition of Generalization Error

For a model $\hat{f}$ trained on dataset $\mathcal{D}$, the expected test error on a new point $(x, y)$ decomposes as:

$$\mathbb{E}[(y - \hat{f}(x))^2] = \underbrace{\text{Bias}^2[\hat{f}(x)]}_{\text{systematic error}} + \underbrace{\text{Var}[\hat{f}(x)]}_{\text{estimation error}} + \underbrace{\sigma^2_\epsilon}_{\text{irreducible noise}}$$

- **Bias** = $\mathbb{E}[\hat{f}(x)] - f^*(x)$: error from wrong assumptions in the model class (underfitting).
- **Variance** = $\mathbb{E}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]$: sensitivity to fluctuations in the training set (overfitting).
- **Irreducible noise** $\sigma^2_\epsilon$: noise in $y$ that no model can predict.

The bias-variance tradeoff is not a law of nature — modern overparameterized models (deep networks, large ensembles) can achieve low bias *and* low variance through the **double descent** phenomenon: as model complexity increases past the interpolation threshold, variance decreases again. But for classical models, the tradeoff is real and guides model selection.

Cross-validation is the empirical tool for estimating the combined bias + variance without knowing $f^*$.

---

## The Train / Validation / Test Split

Three distinct roles:

- **Training set:** Fit model parameters.
- **Validation set:** Tune hyperparameters and select among model candidates.
- **Test set:** One-time final evaluation of the chosen model. Must not influence any decision.

**The test set must never be touched until the final evaluation.** Iterating on a model after seeing test performance — even informally — inflates the apparent generalization error because you are effectively tuning to the test set. This is called **test set contamination**.

Typical splits: 60/20/20 or 70/15/15 for large datasets. For small datasets, the held-out validation set wastes too much data — use cross-validation instead.

---

## K-Fold Cross-Validation

Partition the training data into $K$ equal-sized folds. For each fold $k$:
1. Train on the remaining $K-1$ folds.
2. Evaluate on fold $k$.

The CV estimate of generalization error is:
$$\hat{\text{Err}}_{\text{CV}} = \frac{1}{K} \sum_{k=1}^K \text{Err}_k$$

**Bias-variance tradeoff in $K$:**
- **Small $K$ (e.g., $K=5$):** Each training set is ~80% of data — less bias from using less data. But fewer folds means higher variance in the error estimate.
- **Large $K$ (e.g., $K=N$, LOOCV):** Each model sees almost all the data — near-unbiased estimate of true generalization error. But $N$ models must be trained (expensive), and the $N$ test errors are highly correlated (each fold differs by only one point), producing a high-variance estimate.

The practical sweet spot is **$K = 5$ or $K = 10$**, which balance bias and variance with manageable compute.

---

## Leave-One-Out Cross-Validation (LOOCV)

The limiting case $K = N$: each fold is a single sample. For certain models (e.g., linear regression, kernel smoothers), LOOCV has a closed-form shortcut:

$$\text{LOOCV} = \frac{1}{N}\sum_{i=1}^N \left(\frac{y_i - \hat{y}_i}{1 - h_{ii}}\right)^2$$

where $h_{ii}$ is the $i$-th diagonal element of the hat matrix $H = X(X^TX)^{-1}X^T$ (the leverage). This avoids re-fitting $N$ models. High-leverage points ($h_{ii}$ close to 1) have an outsized effect on the LOOCV estimate.

---

## Stratified K-Fold

In classification problems with class imbalance, random folds may produce folds with very different class proportions. Stratified K-fold preserves the class distribution in each fold — each fold mirrors the overall class ratio. Always use stratified splitting for classification.

---

## Repeated K-Fold

Run $K$-fold $r$ times with different random splits, then average all $r \times K$ estimates. Reduces the variance of the CV estimate at the cost of $r \times$ the compute. Particularly valuable for small datasets where a single split can produce an unusually lucky or unlucky fold assignment.

---

## Nested Cross-Validation

When hyperparameter tuning is part of the modeling pipeline, standard CV gives an optimistically biased estimate — the model was selected to perform well on the same folds used for evaluation.

Nested CV uses two loops:
- **Outer loop ($K$ folds):** Estimates generalization error.
- **Inner loop ($J$ folds, within each outer train set):** Tunes hyperparameters.

```
for each outer fold k:
    inner CV on outer train set → select best hyperparameters
    retrain on outer train set with best hyperparameters
    evaluate on outer test fold k
```

The outer loop error is an unbiased estimate of the generalization error of the *model selection procedure*, not of any single fixed model. It answers: "if I use this training and tuning procedure on a dataset of this size, what performance can I expect?"

**When to use:** Whenever hyperparameter search is done — grid search, random search, Bayesian optimization.

---

## Time Series Cross-Validation

Standard $K$-fold shuffles data randomly, which causes temporal leakage — future data trains models that predict the past. For time series, use **walk-forward validation**:

```
Train: [t1 ... t_k]     Test: [t_{k+1} ... t_{k+h}]
Train: [t1 ... t_{k+h}] Test: [t_{k+h+1} ... t_{k+2h}]
...
```

Variants:
- **Expanding window:** Training set grows with each fold (uses all available history).
- **Rolling window:** Training set has fixed size (emphasizes recent patterns, forgets old ones).

The gap between train and test windows is sometimes set to the forecast horizon $h$ to avoid leakage from features that look forward in time.

---

## Group K-Fold

When observations are not independent — multiple rows from the same patient, user, or store — folds must respect group boundaries. Splitting randomly would allow the model to learn from training-set rows of a group and be evaluated on test-set rows of the same group, producing an optimistically biased estimate (the model has effectively seen that entity).

**GroupKFold** ensures all rows from a given group appear in the same fold. The evaluation then reflects performance on *unseen groups*, which is the correct generalization target.

---

## Hyperparameter Tuning Strategies

### Grid Search
Exhaustive search over a predefined Cartesian product of hyperparameter values. Guaranteed to find the best combination in the grid, but scales exponentially with the number of parameters.

### Random Search
Sample hyperparameters from a defined distribution. For a fixed compute budget, random search typically outperforms grid search when only a few hyperparameters truly matter (Bergstra & Bengio, 2012) — grid search wastes evaluations repeating the same values on unimportant dimensions.

### Bayesian Optimization
Fit a surrogate model (usually a Gaussian process or tree-structured Parzen estimator) over the hyperparameter space, using it to guide search toward promising regions. Balances **exploitation** (evaluate near current best) and **exploration** (evaluate uncertain regions). Significantly more sample-efficient than random search for expensive-to-evaluate models.

The acquisition function (Expected Improvement, Upper Confidence Bound) formalizes the explore/exploit tradeoff:
$$\text{EI}(\lambda) = \mathbb{E}[\max(f(\lambda) - f^+, 0)]$$
where $f^+$ is the best observed performance so far.

### Successive Halving / Hyperband
Allocate a small budget to many configurations, keep only the top fraction, increase budget, repeat. Eliminates bad hyperparameters early without fully training them. Hyperband wraps successive halving over multiple bracket sizes to handle the tradeoff between few-expensive vs. many-cheap evaluations.

---

## Evaluation Metrics for Regression

| Metric | Formula | Notes |
|---|---|---|
| MAE | $\frac{1}{n}\sum \|y_i - \hat{y}_i\|$ | Robust to outliers, in original units |
| MSE | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Penalizes large errors heavily |
| RMSE | $\sqrt{\text{MSE}}$ | Same units as $y$, dominated by large errors |
| MAPE | $\frac{1}{n}\sum \left\|\frac{y_i - \hat{y}_i}{y_i}\right\|$ | Scale-free, undefined when $y_i = 0$, biased for asymmetric errors |
| $R^2$ | $1 - \frac{\text{SS}_\text{res}}{\text{SS}_\text{tot}}$ | Fraction of variance explained; can be negative |
| Adjusted $R^2$ | $1 - (1-R^2)\frac{n-1}{n-p-1}$ | Penalizes adding uninformative features |

**$R^2$ interpretation:** $R^2 = 0$ means the model does no better than predicting the mean $\bar{y}$ for every point. $R^2 < 0$ means it is actively worse than that — possible on test data even if training $R^2 > 0$.

---

## Statistical Significance of Performance Differences

When comparing two models $A$ and $B$ using $K$-fold CV, the $K$ fold scores are paired observations (same data, different models). Use a **paired $t$-test**:

$$t = \frac{\bar{d}}{s_d / \sqrt{K}}, \qquad \bar{d} = \frac{1}{K}\sum_{k=1}^K (\text{Err}^A_k - \text{Err}^B_k)$$

**Caveat (Dietterich, 1998):** The $K$ fold scores are not truly independent (training sets overlap), so the paired $t$-test is anti-conservative — it underestimates the p-value. The **5×2 CV test** uses 5 repetitions of 2-fold CV with a corrected variance estimate to achieve better Type I error control.

For large test sets, **McNemar's test** compares classifiers on the same held-out set using the number of cases where one model is right and the other is wrong — a sign test on disagreements rather than a parametric test on scores.

---

## Learning Curves

Plot training and validation error as a function of training set size $n$. Diagnoses bias/variance:

- **High bias (underfitting):** Both training and validation error converge to a high value. Adding more data will not help — the model class is too simple.
- **High variance (overfitting):** Large gap between training (low) and validation (high) error. Adding more data will help — the gap narrows as $n$ grows.
- **Well-fitted:** Both curves converge close together to a low value.

**Validation curves** are the analogous plot against a hyperparameter (e.g., tree depth, regularization strength) — they reveal the optimal regularization level and the model's sensitivity to it.

---

## The Generalization Gap and Overfitting Taxonomy

| Symptom | Cause | Fix |
|---|---|---|
| High train error, high val error | Underfitting (high bias) | More complex model, better features, less regularization |
| Low train error, high val error | Overfitting (high variance) | More data, regularization, simpler model, dropout |
| Low train error, low val error, high test error | Test set contamination or distribution shift | Stricter data hygiene; monitor distribution |
| Val error improves then plateaus | Optimal stopping point | Early stopping |
| Val error improves then *worsens* | Overfitting to validation set via hyperparameter tuning | Nested CV or hold-out test set |


# Bias-Variance Decomposition

The bias-variance decomposition is a precise mathematical account of *why* a learning algorithm generalizes poorly. It attributes expected prediction error to three independent sources and makes the consequences of model complexity concrete.

---

## Setup

Let the true data-generating process be:
$$y = f^*(x) + \epsilon, \qquad \epsilon \sim (0, \sigma^2_\epsilon)$$

where $f^*$ is the true function and $\epsilon$ is irreducible noise with zero mean and variance $\sigma^2_\epsilon$.

A learning algorithm $\mathcal{A}$ trained on a dataset $\mathcal{D}$ of size $n$ produces an estimator $\hat{f}_\mathcal{D}(x)$. We consider the algorithm as a random process: randomness comes from which training set $\mathcal{D}$ was drawn from the data-generating distribution.

The **expected prediction error** at a fixed point $x$ — averaged over all possible training sets and all noise realizations — is:

$$\text{Err}(x) = \mathbb{E}_{\mathcal{D}, \epsilon}\left[(y - \hat{f}_\mathcal{D}(x))^2\right]$$

---

## The Derivation

Add and subtract $\mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]$ (the expected prediction, averaged over training sets):

$$\text{Err}(x) = \mathbb{E}\left[\left(y - \mathbb{E}[\hat{f}] + \mathbb{E}[\hat{f}] - \hat{f}_\mathcal{D}\right)^2\right]$$

Expand and use $\mathbb{E}[\epsilon] = 0$ and independence of $\epsilon$ from $\mathcal{D}$:

$$\boxed{\text{Err}(x) = \underbrace{\left(f^*(x) - \mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}_\mathcal{D}\left[\left(\hat{f}_\mathcal{D}(x) - \mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma^2_\epsilon}_{\text{Irreducible noise}}}$$

The cross-terms vanish: the $\epsilon$-noise cross-term vanishes by $\mathbb{E}[\epsilon] = 0$; the bias-variance cross-term vanishes because $\mathbb{E}[\hat{f} - \mathbb{E}[\hat{f}]] = 0$ by definition.

---

## Each Term in Detail

### Bias²

$$\text{Bias}^2(x) = \left(f^*(x) - \mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]\right)^2$$

The squared difference between the true function and the *average* prediction of the algorithm. Bias is a property of the **model class and algorithm**, not of any single fitted model.

A biased algorithm is systematically wrong — no matter how much data it sees, it cannot reproduce $f^*$ because the hypothesis class does not contain it. A linear model applied to a quadratic truth has irreducible bias.

**High-bias scenarios:**
- Model class too simple (linear model for nonlinear data)
- Excessive regularization (large $\lambda$ in ridge/lasso forces coefficients toward zero)
- Too few features, insufficient model capacity

### Variance

$$\text{Var}(x) = \mathbb{E}_\mathcal{D}\left[\left(\hat{f}_\mathcal{D}(x) - \mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]\right)^2\right]$$

The expected squared deviation of the prediction from its own mean across training sets. Variance measures how sensitive the algorithm is to the particular training set drawn.

A high-variance algorithm produces wildly different models for different training samples. It is fitting noise alongside signal. Variance is a property of the **interaction between the algorithm and the training set size**.

**High-variance scenarios:**
- Model class too complex (deep trees, high-degree polynomials)
- Insufficient regularization
- Small training set $n$ — with few samples, sampling noise dominates

### Irreducible Noise

$$\sigma^2_\epsilon = \mathbb{E}[\epsilon^2]$$

The variance of the noise in $y$ itself. No model can predict this — it is fundamentally unknowable from $x$. It sets a floor on achievable error.

---

## The Classical Tradeoff

As model complexity increases:

| Complexity | Bias | Variance | Total Error |
|---|---|---|---|
| Very low (underfitting) | High | Low | High |
| Optimal | Low | Low | Minimum |
| Very high (overfitting) | Near zero | High | High |

The optimal model minimizes the sum $\text{Bias}^2 + \text{Variance}$. This tradeoff is the theoretical justification for regularization: regularization deliberately introduces bias to reduce variance by more, lowering total error.

---

## Estimating Bias and Variance Empirically

You cannot observe $\text{Bias}^2$ or $\text{Variance}$ directly from a single fitted model — doing so requires knowing $f^*$. But you can estimate them by simulation:

1. Generate $B$ bootstrap datasets $\mathcal{D}_1, \ldots, \mathcal{D}_B$ from the training data.
2. For each, fit $\hat{f}_b$.
3. At a test point $x$:

$$\widehat{\text{Bias}}^2(x) = \left(f^*(x) - \frac{1}{B}\sum_b \hat{f}_b(x)\right)^2$$

$$\widehat{\text{Var}}(x) = \frac{1}{B}\sum_b \left(\hat{f}_b(x) - \frac{1}{B}\sum_{b'}\hat{f}_{b'}(x)\right)^2$$

In practice, $f^*(x)$ is approximated by the noiseless ground truth (in simulations) or replaced by $y$ (which adds $\sigma^2_\epsilon$ to the bias estimate).

---

## Effect of Training Set Size $n$

For most algorithms:

$$\text{Bias}(n) \approx \text{const} \quad \text{(does not decrease with more data for a fixed model class)}$$

$$\text{Variance}(n) \propto \frac{1}{n} \quad \text{(decreases as more data constrains the fit)}$$

This has a critical practical implication: **more data reduces variance but not bias**. If a model is underfitting (high bias), collecting more data will not help — you need to change the model class. If it is overfitting (high variance), more data is often the most reliable fix.

---

## The Decomposition for Classification (0-1 Loss)

The squared-error decomposition does not transfer directly to 0-1 loss because the loss is not quadratic. Domingos (2000) and Kohavi & Wolpert (1996) derived analogous decompositions for classification:

$$\text{Err} = \text{Noise} + \text{Bias} + \text{Variance}$$

where:
- **Noise** = Bayes error, the irreducible error of the optimal classifier.
- **Bias** = excess error from the algorithm systematically predicting the wrong class (the *main prediction* differs from the Bayes-optimal prediction).
- **Variance** = contribution from the algorithm's prediction *changing* across training sets — variance *helps* when it causes the algorithm to occasionally predict the correct class even when the main prediction is wrong (variance can be *negative* in its net effect on error).

This is a subtler result than the regression case: in classification, variance is not always bad. A high-variance algorithm that is biased can sometimes correct its own bias by varying its prediction, which can *reduce* classification error below what bias alone would suggest.

---

## Double Descent

Classical bias-variance theory predicts a U-shaped test error curve as model complexity grows. Modern overparameterized models (neural networks with more parameters than training points) show a second descent:

```
Error
  |  \         /‾‾‾\        /
  |   \       /      \      /
  |    \_____/        \____/
  |
  +------|--------|--------|------> Complexity
      classical  interpolation  overparameterized
       regime    threshold      regime
```

At the **interpolation threshold** (model can exactly fit training data), test error peaks. Beyond it, as the model is further overparameterized, test error *falls again* — sometimes below the classical optimum.

**Why:** Gradient descent on overparameterized networks finds the *minimum-norm* interpolating solution. Among all solutions that fit the training data perfectly, this is the smoothest one — and smooth interpolants often generalize well. This is related to the **implicit regularization** of gradient descent and the effective prior imposed by the network architecture.

Double descent does not invalidate bias-variance thinking — it extends it. In the overparameterized regime, variance decreases because the model is constrained to be smooth, not because it has fewer parameters.

---

## Regularization as Bias-Variance Control

Every regularization method is a lever on the bias-variance tradeoff:

| Regularization | Mechanism | Effect |
|---|---|---|
| $\ell_2$ (Ridge) | Shrinks coefficients toward zero | Reduces variance, adds bias |
| $\ell_1$ (Lasso) | Sparsifies coefficients | Reduces variance via feature selection |
| Dropout | Randomly zeroes activations during training | Ensemble-like variance reduction |
| Early stopping | Halts before full convergence | Limits effective model complexity |
| Data augmentation | Increases effective $n$ | Reduces variance |
| Ensemble (bagging) | Averages predictions over $B$ models | Reduces variance without increasing bias |
| Ensemble (boosting) | Iteratively reduces residuals | Reduces bias, can increase variance |

**Bagging** is the clearest example of pure variance reduction. The variance of the mean of $B$ i.i.d. random variables with variance $\sigma^2$ is $\sigma^2/B$. Bootstrap aggregation approximates this — each tree has high variance, but their average has low variance. Bias is unchanged because each tree is unbiased (in expectation, deep trees fit $f^*$ well).

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^B \hat{f}_b\right) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$$

where $\rho$ is the pairwise correlation between trees. As $B \to \infty$, the second term vanishes and variance is bounded below by $\rho \sigma^2$ — which is why random forests actively decorrelate trees (random feature subsets at each split), reducing $\rho$ and pushing variance lower.


# Classification Evaluation

## The Confusion Matrix

For a binary classifier with threshold $\tau$, every prediction falls into one of four cells:

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | TP | FN |
| **Actually Negative** | FP | TN |

- **TP** — correct positive prediction
- **FP** — Type I error (false alarm)
- **FN** — Type II error (miss)
- **TN** — correct negative prediction

The total population: $N = \text{TP} + \text{FP} + \text{FN} + \text{TN}$.

---

## Derived Metrics — A Unified View

Every scalar metric is a function of these four counts. Understanding their denominators is the key to choosing the right one.

### Accuracy

$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{N}$$

Fraction of all predictions that are correct. **Misleading under class imbalance** — a classifier that always predicts "negative" on a 99/1 dataset achieves 99% accuracy while being useless.

---

### Precision and Recall (the fundamental trade-off)

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} \qquad \text{Recall (Sensitivity, TPR)} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

- **Precision** conditions on *predicted* positives: "of what I flagged, how much was real?"
- **Recall** conditions on *actual* positives: "of what was real, how much did I catch?"

They are in tension. Raising the threshold $\tau$ increases precision (fewer FP) but decreases recall (more FN are missed). This is not a flaw — it reflects the genuine cost trade-off in the problem.

**Precision-Recall curve:** Sweep $\tau$ from 0 to 1 and plot Recall (x) vs. Precision (y). **Area Under the PR Curve (AUPRC / Average Precision)** summarizes the curve into a single number. Preferred over ROC-AUC when positives are rare, because it is *not* influenced by the large number of TNs.

---

### Specificity and False Positive Rate

$$\text{Specificity (TNR)} = \frac{\text{TN}}{\text{TN} + \text{FP}} \qquad \text{FPR} = 1 - \text{Specificity} = \frac{\text{FP}}{\text{FP} + \text{TN}}$$

Specificity conditions on *actual negatives*. The FPR is the fraction of real negatives you incorrectly flagged.

---

### F-scores

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

The harmonic mean family. $\beta$ controls the weighting:
- $\beta = 1$ ($F_1$): equal weight — appropriate when FP and FN costs are symmetric.
- $\beta > 1$: favors recall — use when missing a positive is more costly (e.g., disease screening).
- $\beta < 1$: favors precision — use when a false alarm is more costly (e.g., spam filter).

**Why harmonic mean?** The harmonic mean is dominated by whichever of precision/recall is lower. A classifier that achieves precision=1, recall=0.01 gets $F_1 \approx 0.02$, correctly penalizing the near-useless recall. Arithmetic mean would give 0.5, hiding the failure.

---

### Matthews Correlation Coefficient (MCC)

$$\text{MCC} = \frac{\text{TP} \cdot \text{TN} - \text{FP} \cdot \text{FN}}{\sqrt{(\text{TP}+\text{FP})(\text{TP}+\text{FN})(\text{TN}+\text{FP})(\text{TN}+\text{FN})}}$$

$\text{MCC} \in [-1, 1]$. It is the **Pearson correlation** between the binary true labels and binary predicted labels, taking all four quadrants into account simultaneously. It is widely considered the most informative single metric for binary classification under imbalance because it is only high when all four counts are proportionally reasonable.

---

## ROC Curve and AUC

**The ROC curve** plots TPR (Recall) on the y-axis vs. FPR on the x-axis as the threshold $\tau$ sweeps from 1 to 0.

$$\text{TPR}(\tau) = P(\hat{p} > \tau \mid Y=1), \qquad \text{FPR}(\tau) = P(\hat{p} > \tau \mid Y=0)$$

**Key points:**
- $(0, 0)$: threshold so high nothing is predicted positive.
- $(1, 1)$: threshold so low everything is predicted positive.
- $(0, 1)$: perfect classifier.
- Diagonal line $\text{TPR} = \text{FPR}$: random guessing (uninformative model).

**AUC-ROC** — Area Under the ROC Curve:

$$\text{AUC} = \int_0^1 \text{TPR}(\text{FPR}) \, d(\text{FPR})$$

**Probabilistic interpretation (Wilcoxon-Mann-Whitney):**

$$\text{AUC} = P(\hat{p}_{\text{pos}} > \hat{p}_{\text{neg}})$$

AUC is the probability that a randomly chosen positive example is scored higher than a randomly chosen negative example. This makes it a **ranking metric** — it measures discrimination ability regardless of the chosen threshold or class balance.

**AUC = 0.5:** model has no discriminative power (equivalent to random).  
**AUC = 1.0:** perfect ranking.

**Why AUC is insensitive to class imbalance:** The TPR denominator conditions on actual positives; the FPR denominator conditions on actual negatives. The two populations are evaluated independently, so changing the ratio of positives to negatives in the dataset does not move the curve.

---

## ROC vs. PR Curve — When to Use Which

| | ROC / AUC | PR / AUPRC |
|---|---|---|
| Class balance | Robust to imbalance | Sensitive to imbalance |
| Interpretation | Ranking / discrimination | Retrieval quality |
| TN contribution | Yes (via FPR) | No |
| Use when | Both classes matter equally | Positives are rare and important |

When positives are very rare, TN is enormous. Because FPR = FP/(FP+TN), even many FPs look like a small FPR — the ROC curve looks optimistic. The PR curve avoids this by never looking at TNs.

---

## Calibration vs. Discrimination

A model can have excellent AUC (good ranking) but poor **calibration** — the predicted probabilities don't correspond to actual frequencies.

**Reliability diagram (calibration curve):** Bin predictions by $\hat{p}$, plot mean $\hat{p}$ (x) vs. observed positive rate (y). A perfectly calibrated model lies on $y = x$.

**Expected Calibration Error (ECE):**
$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N} \left| \overline{y}_{B_b} - \overline{\hat{p}}_{B_b} \right|$$

Weighted mean absolute deviation between confidence and accuracy per bin.

Calibration methods:
- **Platt scaling:** fit a logistic regression on top of the raw scores.
- **Isotonic regression:** non-parametric monotone recalibration.
- **Temperature scaling:** divide logits by a single scalar $T$ (common in neural nets).

---

## Multi-class Extension

For $K$ classes the confusion matrix becomes $K \times K$. Row = true class, column = predicted class. Diagonal = correct predictions.

Metrics generalize via **averaging strategies:**

- **Macro:** compute metric per class, take unweighted mean. Treats all classes equally regardless of support.
- **Micro:** aggregate TP, FP, FN across all classes before computing. Dominated by the majority class.
- **Weighted:** per-class metric weighted by support (sample count). Balanced between macro and micro.

For multi-class ROC, the standard approach is **one-vs-rest** (OvR): compute AUC for each class against all others, then average (macro or weighted).

---

## Summary: Choosing a Metric

| Scenario | Recommended metric |
|---|---|
| Balanced classes, symmetric costs | Accuracy, $F_1$, AUC |
| Imbalanced, rare positives matter | AUPRC, $F_\beta$ ($\beta > 1$), MCC |
| False alarms are costly | Precision, $F_\beta$ ($\beta < 1$) |
| Threshold-free ranking quality | AUC-ROC |
| Probabilistic outputs need trust | ECE, calibration curve |
| Multi-class, all classes matter equally | Macro $F_1$, macro AUC |


# Missing Values

## The Three Missingness Mechanisms (Rubin, 1976)

The most important question about missing data is *why* it is missing. Rubin's taxonomy is the theoretical foundation for every decision that follows.

Let $Y$ be the full data matrix, $R$ a binary missingness indicator ($R_{ij} = 1$ if $Y_{ij}$ is observed), and $\psi$ parameters governing the missingness process.

### MCAR — Missing Completely At Random

$$P(R \mid Y, \psi) = P(R \mid \psi)$$

Missingness is independent of both observed and unobserved values. The missing data is a simple random subsample of the full data.

**Example:** A sensor randomly fails with probability 0.01, regardless of what it would have measured.

**Consequence:** Complete-case analysis (dropping rows) is unbiased but inefficient. Any imputation method works. MCAR is testable via Little's MCAR test, but rarely holds in practice.

---

### MAR — Missing At Random

$$P(R \mid Y, \psi) = P(R \mid Y_{\text{obs}}, \psi)$$

Missingness depends only on *observed* values, not on the missing value itself. Once you condition on what you can see, the missingness is random.

**Example:** Men are less likely to report income, but among men and women with the same age and education, income missingness is random with respect to the actual income value.

**Consequence:** Likelihood-based methods (EM algorithm, multiple imputation) are valid under MAR. Complete-case analysis is biased. MAR is *not* directly testable — you cannot distinguish MAR from MNAR from the observed data alone.

---

### MNAR — Missing Not At Random

$$P(R \mid Y, \psi) = P(R \mid Y_{\text{obs}}, Y_{\text{miss}}, \psi)$$

Missingness depends on the unobserved value itself. The most dangerous case.

**Example:** Patients with very high blood pressure are more likely to skip a check-up, so the most extreme blood pressure values are systematically absent.

**Consequence:** No standard imputation method corrects for MNAR without modeling the missingness mechanism explicitly (selection models, pattern-mixture models). Sensitivity analysis is essential. MNAR is also untestable from data alone.

---

## Imputation Methods

### Deletion

**Complete-case analysis (listwise deletion):** Drop any row with at least one missing value.
- Unbiased only under MCAR.
- Can lose enormous amounts of data (if 10 features each have 5% missing, ~40% of rows have at least one missing value).

**Pairwise deletion:** Use all available pairs for each computation (e.g., covariance matrices). Produces inconsistent estimates — the resulting matrix may not be positive semi-definite.

---

### Single Imputation

Replace each missing value with a single estimate. Simple but underestimates uncertainty because the imputed value is treated as if it were observed.

**Mean/median/mode imputation:**
- Mean: minimizes squared error but shrinks variance. The imputed variable's variance becomes $\sigma^2(1 - f)$ where $f$ is the fraction missing — always an underestimate.
- Median: robust to outliers, better for skewed distributions.
- Mode: appropriate for categorical variables.
- **All three destroy correlations** between the imputed feature and others.

**Regression imputation:** Regress the missing feature on the observed features, predict the missing values. Preserves linear relationships but still underestimates variance (all imputed values lie exactly on the regression surface, adding no residual noise).

**Stochastic regression imputation:** Add a residual draw $\epsilon \sim \mathcal{N}(0, \hat{\sigma}^2)$ to the regression prediction. Restores marginal variance but still treats imputed values as known.

---

### Multiple Imputation (MI)

The principled solution to the variance-underestimation problem. Generate $m$ complete datasets, each with different imputed values drawn from the *posterior predictive distribution* of the missing data. Analyze each dataset separately, then combine results with **Rubin's rules**:

$$\bar{\theta} = \frac{1}{m} \sum_{i=1}^m \hat{\theta}_i$$

$$\text{Var}(\bar{\theta}) = \underbrace{\frac{1}{m}\sum_{i=1}^m \hat{V}_i}_{\text{within-imputation variance}} + \underbrace{\left(1 + \frac{1}{m}\right) \frac{1}{m-1}\sum_{i=1}^m (\hat{\theta}_i - \bar{\theta})^2}_{\text{between-imputation variance}}$$

The between-imputation variance captures uncertainty about the missing values themselves. Under MAR, MI is asymptotically efficient.

**MICE (Multiple Imputation by Chained Equations):** The dominant algorithm. Iteratively fits a separate imputation model per variable using all other variables as predictors, cycling through variables until convergence. Each variable can use the model most appropriate for its type (linear regression, logistic regression, PMM, random forest, etc.).

**Number of imputations $m$:** Rule of thumb: $m \geq 100 \cdot f$ where $f$ is the fraction of incomplete cases. 5–20 was the old standard; modern practice uses 50–100 for stable estimates.

---

### Model-Based Imputation

**EM Algorithm:** Iterates between:
1. **E-step:** Compute the expected sufficient statistics of the complete data, given the observed data and current parameters.
2. **M-step:** Re-estimate parameters by maximizing the expected complete-data log-likelihood.

Converges to the MLE under MAR. Does not produce imputed datasets — it estimates model parameters directly from incomplete data.

**K-Nearest Neighbors (KNN) imputation:** For each missing value, find the $k$ nearest complete rows (by observed features), impute with their mean/mode. Preserves local structure and correlations. Sensitive to the distance metric and scale — **must normalize first.**

**Matrix Factorization / Low-Rank Imputation:** Decompose $Y \approx UV^T$, fit $U, V$ on observed entries (e.g., via alternating least squares or nuclear norm minimization). Exploits global low-rank structure. The theoretical foundation of collaborative filtering (Netflix problem).

$$\min_{U, V} \sum_{(i,j) \in \Omega} (Y_{ij} - U_i V_j^T)^2 + \lambda(\|U\|_F^2 + \|V\|_F^2)$$

where $\Omega$ is the set of observed entries.

**Deep learning (GAIN, MIWAE):** Neural network imputation methods that model the joint distribution $P(Y_{\text{miss}} \mid Y_{\text{obs}})$ directly. Can handle complex nonlinear dependencies but require substantial data and tuning.

---

## Indicator Variables and Missingness as Signal

Under MNAR — and sometimes MAR — the *fact* of missingness is itself informative. Standard practice:

1. Create a binary indicator $M_j = \mathbf{1}[X_j \text{ is missing}]$ for each feature with missingness.
2. Impute $X_j$ by any method.
3. Include both $X_j^{\text{imputed}}$ and $M_j$ in the model.

This allows the model to learn both the value effect and the missingness effect. It is sometimes called the "missing indicator method" and is a pragmatic hedge when you suspect MNAR.

---

## Variance and Bias Under Each Strategy

| Method | Bias (MAR) | Variance | Preserves correlations |
|---|---|---|---|
| Complete-case | Low | High (data loss) | Yes |
| Mean imputation | High | Underestimated | No |
| Regression imputation | Low | Underestimated | Partially |
| Stochastic regression | Low | Correct (marginal) | Partially |
| Multiple imputation (MICE) | Low | Correct | Yes |
| KNN imputation | Low-moderate | Moderate | Locally |
| EM / MLE | None (consistent) | Asymptotically efficient | Yes |

---

## Practical Decision Flow

1. **Quantify missingness:** fraction missing per feature and per row. Visualize with a missingness matrix (e.g., `missingno` library).
2. **Hypothesize the mechanism:** interview domain experts, look for patterns — is missingness correlated with observed values? Run Little's MCAR test as a weak check.
3. **MCAR:** any method works; prefer MI or complete-case for simplicity.
4. **MAR:** use MICE or EM. Avoid mean imputation.
5. **MNAR:** no off-the-shelf fix. Consider: modeling the missingness process explicitly, sensitivity analysis, collecting more data, or using domain knowledge to bound the bias.
6. **High missingness (>40% in a feature):** reconsider whether the feature is useful at all, or whether the missingness indicator alone is the real signal.
7. **Tree-based models:** can handle `NaN` natively (XGBoost, LightGBM learn the optimal split direction for missing values). Imputation is less critical but missingness indicators can still add signal.
8. **Leakage:** fit all imputation models on training data only. In MICE, the imputation models are part of the pipeline and must be re-fit at train time, never on test data.

---

## The Fraction-Missing Information (FMI)

Rubin defined the **fraction of missing information** for a parameter $\theta$:

$$\lambda = \frac{B + B/m}{V_{\text{total}}}$$

where $B$ is the between-imputation variance and $V_{\text{total}}$ is the total variance. $\lambda \in [0,1]$ quantifies how much the missingness degrades inference about $\theta$. A high FMI means more imputations $m$ are needed and the parameter estimate is sensitive to the imputation model assumptions.


# Summary Statistics

Summary statistics compress a distribution into a small number of interpretable scalars. They answer three distinct questions: *where is the distribution centered*, *how spread out is it*, and *what shape does it have*.

---

## Measures of Central Tendency

### Mean (Expected Value)

$$\mu = \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$

The mean minimizes the sum of squared deviations: $\bar{x} = \arg\min_c \sum(x_i - c)^2$. It is the first raw moment of the distribution. Sensitive to outliers because every value enters with equal weight.

For a random variable: $\mathbb{E}[X] = \int x \, f(x) \, dx$. Key properties:
- **Linearity:** $\mathbb{E}[aX + b] = a\mathbb{E}[X] + b$
- **Not multiplicative in general:** $\mathbb{E}[XY] = \mathbb{E}[X]\mathbb{E}[Y]$ only if $X \perp Y$

### Median

The value $m$ such that $P(X \leq m) \geq 0.5$ and $P(X \geq m) \geq 0.5$. The median minimizes the sum of *absolute* deviations: $m = \arg\min_c \sum|x_i - c|$. Breakdown point of 50% — robust to arbitrarily bad outliers.

For symmetric distributions, mean = median. Skewed distributions pull the mean toward the tail, leaving the median as the better "typical value."

### Mode

The value(s) of highest probability density. Uniquely defined for unimodal distributions, may be non-unique or undefined otherwise. The mode minimizes $\ell_0$ loss (count of mismatches), making it the natural central tendency for categorical data.

### Geometric Mean

$$\bar{x}_g = \left(\prod_{i=1}^n x_i\right)^{1/n} = \exp\left(\frac{1}{n}\sum_{i=1}^n \ln x_i\right)$$

Appropriate when values are multiplicative or span orders of magnitude (financial returns, growth rates, ratios). The geometric mean of returns equals the arithmetic mean of log-returns, which is why log-returns are used in finance.

### Harmonic Mean

$$\bar{x}_h = \frac{n}{\sum_{i=1}^n 1/x_i}$$

Appropriate for rates and ratios (e.g., average speed over a fixed distance). Heavily penalizes small values. The inequality $\bar{x}_h \leq \bar{x}_g \leq \bar{x}$ always holds (AM-GM-HM inequality).

---

## Measures of Spread

### Variance and Standard Deviation

$$\sigma^2 = \text{Var}(X) = \mathbb{E}[(X - \mu)^2] = \mathbb{E}[X^2] - \mu^2$$

The second central moment. Measures average squared deviation from the mean. Standard deviation $\sigma = \sqrt{\sigma^2}$ restores the original units.

**Sample variance** uses $n-1$ (Bessel's correction) to produce an unbiased estimator:
$$s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$$

Bessel's correction compensates for the fact that $\bar{x}$ is estimated from the same data — the deviations $(x_i - \bar{x})$ span only an $(n-1)$-dimensional subspace.

Key properties:
- $\text{Var}(aX + b) = a^2 \text{Var}(X)$
- $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\text{Cov}(X, Y)$
- Variance is not robust — one outlier can inflate it arbitrarily.

### Interquartile Range (IQR)

$$\text{IQR} = Q_3 - Q_1 = F^{-1}(0.75) - F^{-1}(0.25)$$

The spread of the middle 50% of the data. Breakdown point of 25%. The standard rule for outlier detection: values beyond $Q_1 - 1.5 \cdot \text{IQR}$ or $Q_3 + 1.5 \cdot \text{IQR}$ (Tukey's fences).

For a normal distribution, $\text{IQR} \approx 1.35\sigma$, so $\hat{\sigma} = \text{IQR}/1.35$ is a robust estimator of scale.

### Mean Absolute Deviation (MAD)

$$\text{MAD} = \frac{1}{n}\sum_{i=1}^n |x_i - \bar{x}|$$

Uses $\ell_1$ rather than $\ell_2$ loss, so it is less sensitive to outliers than standard deviation. Not to be confused with **Median Absolute Deviation** (also MAD):

$$\text{MAD}_{\text{median}} = \text{median}(|x_i - \text{median}(x)|)$$

The median absolute deviation has a breakdown point of 50% — the most robust scale estimator in common use. For normal data, $\sigma \approx 1.4826 \cdot \text{MAD}_{\text{median}}$.

### Coefficient of Variation (CV)

$$\text{CV} = \frac{\sigma}{\mu}$$

Dimensionless measure of relative spread. Useful for comparing variability across features measured on different scales or with different means. Only meaningful when $\mu > 0$ and the ratio scale is appropriate.

---

## Shape: Skewness and Kurtosis

### Skewness (Third Standardized Moment)

$$\gamma_1 = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^3\right] = \frac{\mu_3}{\sigma^3}$$

where $\mu_3 = \mathbb{E}[(X-\mu)^3]$ is the third central moment.

- $\gamma_1 = 0$: symmetric (e.g., normal)
- $\gamma_1 > 0$: right-skewed (long right tail) — mean > median
- $\gamma_1 < 0$: left-skewed (long left tail) — mean < median

The cube preserves sign, so positive deviations from the mean pull skewness positive. Income and asset prices are classically right-skewed.

### Kurtosis (Fourth Standardized Moment)

$$\gamma_2 = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^4\right] = \frac{\mu_4}{\sigma^4}$$

**Excess kurtosis** = $\gamma_2 - 3$, subtracting the normal distribution's value of 3.

- Excess kurtosis = 0: mesokurtic (normal-like tails)
- Excess kurtosis > 0: **leptokurtic** — heavier tails and sharper peak than normal (e.g., Student's $t$, financial returns). More probability in extremes.
- Excess kurtosis < 0: **platykurtic** — lighter tails, flatter peak (e.g., uniform distribution).

**Common misconception:** Kurtosis is about tail weight, not peak sharpness per se. A distribution can have a sharp peak *and* light tails (platykurtic) if the variance is concentrated near the mean.

---

## Quantiles and the Five-Number Summary

The $p$-th quantile $Q(p) = F^{-1}(p)$ is the value below which $p$ fraction of the data falls.

**Five-number summary:** $\{\min,\ Q_1,\ \text{median},\ Q_3,\ \max\}$ — the basis of the box plot. Together they describe range, spread, and asymmetry without assuming any distributional form.

**Percentile vs. quantile:** percentile is quantile $\times 100$. The 95th percentile = $Q(0.95)$.

---

## Covariance and Correlation

### Covariance

$$\text{Cov}(X, Y) = \mathbb{E}[(X - \mu_X)(Y - \mu_Y)] = \mathbb{E}[XY] - \mu_X \mu_Y$$

Measures the degree to which $X$ and $Y$ vary together. Sign indicates direction; magnitude depends on scale, making raw covariance hard to interpret.

The **covariance matrix** $\Sigma$ for a $d$-dimensional random vector $\mathbf{X}$:
$$\Sigma_{ij} = \text{Cov}(X_i, X_j), \qquad \Sigma = \mathbb{E}[(\mathbf{X} - \boldsymbol{\mu})(\mathbf{X} - \boldsymbol{\mu})^T]$$

$\Sigma$ is always symmetric positive semi-definite. Its eigenvalues are the variances along the principal axes (the basis of PCA).

### Pearson Correlation

$$\rho(X, Y) = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y} \in [-1, 1]$$

The covariance normalized to be scale-free. $|\rho| = 1$ iff $Y = aX + b$ (perfect linear relationship). $\rho = 0$ implies no *linear* relationship — but $X$ and $Y$ may still be strongly dependent nonlinearly.

### Rank Correlations

**Spearman's $\rho_s$:** Pearson correlation applied to the ranks of $X$ and $Y$. Measures monotone (not just linear) association. Robust to outliers and appropriate for ordinal data.

**Kendall's $\tau$:** Based on concordant vs. discordant pairs:
$$\tau = \frac{C - D}{\binom{n}{2}}$$
where $C$ = concordant pairs, $D$ = discordant pairs. More interpretable as a probability: $\tau = P(\text{concordant}) - P(\text{discordant})$. More robust than Spearman's $\rho_s$ for small samples and tied values.

---

## Moments and the Moment Generating Function

The $k$-th **raw moment**: $\mu'_k = \mathbb{E}[X^k]$

The $k$-th **central moment**: $\mu_k = \mathbb{E}[(X - \mu)^k]$

| $k$ | Central moment | Common name |
|---|---|---|
| 1 | 0 (by definition) | — |
| 2 | $\sigma^2$ | Variance |
| 3 | $\mu_3$ | (related to skewness) |
| 4 | $\mu_4$ | (related to kurtosis) |

The **Moment Generating Function (MGF)**: $M_X(t) = \mathbb{E}[e^{tX}]$

Differentiating: $\mathbb{E}[X^k] = M_X^{(k)}(0)$. The MGF uniquely determines the distribution (when it exists in a neighborhood of 0) and is the tool of choice for proving the Central Limit Theorem and computing distributions of sums.

---

## Quick Reference: Robustness

| Statistic | Breakdown point | Sensitive to outliers |
|---|---|---|
| Mean | 0% | Yes |
| Variance / SD | 0% | Yes |
| Median | 50% | No |
| MAD (median-based) | 50% | No |
| IQR | 25% | No |
| Trimmed mean ($\alpha$%) | $\alpha$% | Partially |
| Winsorized mean | Configurable | Partially |

A **trimmed mean** drops the top and bottom $\alpha\%$ of values before averaging — a smooth interpolation between mean (0% trimming) and median (50% trimming).

A **Winsorized mean** replaces extremes with the boundary values rather than removing them, preserving sample size while limiting outlier influence.
